# GenWildSplat Diagnostic Evaluation Notebook

This notebook documents a compact inference-time diagnostic study of GenWildSplat. It follows the public implementation, sets up the runtime environment, downloads the released checkpoint, runs a reproduction sanity check, evaluates a small Mip-NeRF 360 diagnostic set, compares blank masks against YOLO-generated masks, and traces the light-code pathway for future experimentation.

The code cells are kept unchanged from the working notebook. Markdown cells are added only to make the workflow easier to read and audit.

## Notebook structure

1. Kaggle runtime and repository setup
2. CUDA, PyTorch, and dependency installation
3. Checkpoint download and memory-related preprocessing patch
4. Reproduction sanity check on the public Oidor Chapel example
5. Diagnostic scene selection from Mip-NeRF 360
6. Probe 1: diagnostic generalization behavior
7. Probe 2: occlusion-mask sensitivity
8. Attempted probe: light-code pathway tracing


## 1. Kaggle runtime and repository setup

The first cells inspect the Kaggle runtime, clone the GenWildSplat repository, and review the repository structure and dependency information. These cells establish the working directory and confirm that the expected source files are available.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%cd /kaggle/working
!git clone https://github.com/Vinayak-VG/GenWildSplat.git

In [ ]:
%cd /kaggle/working/GenWildSplat
!ls -la

In [ ]:
# What does the repo say to do, and what does it depend on?
!cat README.md
!echo "----- requirements -----"
!cat requirements.txt 2>/dev/null || echo "no requirements.txt"
!echo "----- top-level structure -----"
!ls -R . | head -60

## 2. CUDA and PyTorch compatibility check

The public implementation depends on a specific PyTorch/CUDA stack. The next cell records the default Kaggle environment before installing the tested stack required for the GenWildSplat dependencies.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
!nvcc --version 2>/dev/null || echo "no nvcc"
!gcc --version | head -1

## 3. Environment installation

The following cells install the PyTorch 2.4.0 + CUDA 12.4 stack and create a separate Conda environment for the GenWildSplat dependencies. In the original run, the notebook was restarted after the first PyTorch stack change; this is kept explicit because it affects reproducibility on Kaggle.


In [ ]:
# --- match the repo's tested stack (torch 2.4.0 + cu124) ---
#this code will take around 3 minutes to run 
import os
os.chdir('/kaggle/working/GenWildSplat')

# 1. Overwrite Kaggle's torch 2.10 with the repo's 2.4.0+cu124
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu124

# 2. Verify the downgrade actually took BEFORE going further
import torch
print("torch now:", torch.__version__, "| cuda:", torch.version.cuda)

### Restart note

After the previous installation cell, restart the notebook session and clear outputs if running from scratch on Kaggle. Then continue with the verification cell below.


In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)

In [ ]:
%%bash
cd /kaggle/working
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
bash miniconda.sh -b -p /kaggle/working/miniconda
/kaggle/working/miniconda/bin/conda --version

In [ ]:
import subprocess
result = subprocess.run(
    ["/kaggle/working/miniconda/bin/conda", "create", "-y", "-n", "genwild",
     "python=3.10", "-c", "conda-forge", "--override-channels"],
    capture_output=True, text=True
)
print("RETURN CODE:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

In [ ]:
!/kaggle/working/miniconda/envs/genwild/bin/python --version

In [ ]:
!/kaggle/working/miniconda/bin/conda install -y -n genwild -c conda-forge pip --override-channels

In [ ]:
!/kaggle/working/miniconda/envs/genwild/bin/python -m pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!/kaggle/working/miniconda/envs/genwild/bin/python -c "import torch; print('env torch:', torch.__version__, torch.version.cuda)"

In [ ]:
!/kaggle/working/miniconda/envs/genwild/bin/python -m pip install "https://github.com/nerfstudio-project/gsplat/releases/download/v1.5.3/gsplat-1.5.3%2Bpt24cu124-cp310-cp310-linux_x86_64.whl"

In [ ]:
!cd /kaggle/working/GenWildSplat && /kaggle/working/miniconda/envs/genwild/bin/python -m pip install -r requirements.txt --find-links https://data.pyg.org/whl/torch-2.4.0+cu124.html

In [ ]:
!cd /kaggle/working/GenWildSplat && /kaggle/working/miniconda/envs/genwild/bin/python -m pip install -e SyncFix --no-deps

In [ ]:
check_code = '''
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
import gsplat
print("gsplat", gsplat.__version__)
import torch_scatter
print("torch_scatter ok")
import xformers
print("xformers", xformers.__version__)
import diffusers, transformers, kornia, lpips
print("all core imports ok")
'''

with open('/kaggle/working/check_env.py', 'w') as f:
    f.write(check_code)

!/kaggle/working/miniconda/envs/genwild/bin/python /kaggle/working/check_env.py

## 4. Checkpoint download, extraction, and preprocessing patch

The next cells download the released GenWildSplat checkpoint archive, verify its contents, move the checkpoint files into the expected directory, and patch the preprocessing resolution from 448 to 392 pixels to fit the available GPU memory in the Kaggle environment. The resolution patch is part of the local reproduction setup and should be considered when interpreting absolute metric values.


In [ ]:
# upgrade gdown first (gets --fuzzy support + fewer Drive quirks)
!/kaggle/working/miniconda/envs/genwild/bin/python -m pip install -U gdown -q

# download by file ID directly (no --fuzzy needed)
!cd /kaggle/working/GenWildSplat && /kaggle/working/miniconda/envs/genwild/bin/python -m gdown 1LNK4_NqZOjGw4OKt3cLrsRW0tH-pfnov -O checkpoint_download.zip

In [ ]:
!ls -la /kaggle/working/GenWildSplat/*.zip /kaggle/working/GenWildSplat/checkpoint_download* 2>/dev/null
!file /kaggle/working/GenWildSplat/checkpoint_download.zip 2>/dev/null

In [ ]:
!/kaggle/working/miniconda/envs/genwild/bin/python -m zipfile -l /kaggle/working/GenWildSplat/checkpoint_download.zip | head -30

In [ ]:
import os

# 1. Delete the failed partial copy in checkpoint/ (reclaim whatever it wrote)
partial = "/kaggle/working/GenWildSplat/checkpoint/model.safetensors"
if os.path.exists(partial):
    os.remove(partial)
    print("removed partial copy")

# 2. Delete the zip NOW — we already extracted it, don't need it. Frees ~5GB.
zip_path = "/kaggle/working/GenWildSplat/checkpoint_download.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
    print("removed zip")

# 3. Check disk
!df -h /kaggle/working

In [ ]:
import shutil, os

extract_to = "/kaggle/working/GenWildSplat/checkpoint_extracted"
ckpt_dir = "/kaggle/working/GenWildSplat/checkpoint"

for fname in ["model.safetensors", "yolov8x-seg.pt"]:
    src = os.path.join(extract_to, fname)
    dst = os.path.join(ckpt_dir, fname)
    if os.path.exists(src):
        shutil.move(src, dst)   # move, not copy — no duplication
        print(f"moved {fname}")

# confirm checkpoint/ has all three
print("\n--- checkpoint/ ---")
for f in os.listdir(ckpt_dir):
    size = os.path.getsize(os.path.join(ckpt_dir, f))
    print(f"{f}: {size/1e6:.1f} MB")

# clean up the now-empty extraction folder + its MACOSX cruft
shutil.rmtree(extract_to, ignore_errors=True)
print("\ncleaned extraction folder")

In [ ]:
!ls -la /kaggle/working/GenWildSplat/examples/Oidor_Chapel/

In [ ]:
!grep -n "def process_image" /kaggle/working/GenWildSplat/src/utils/image.py

In [ ]:
import re

path = "/kaggle/working/GenWildSplat/src/utils/image.py"

# backup
import shutil
shutil.copy(path, path + ".bak")

with open(path) as f:
    src = f.read()

# Replace ONLY within process_image: the four 448 literals.
# process_image is the last function; safe to replace 448 -> 392 globally
# since 448 only appears inside process_image here.
new = src.replace("448", "392")

with open(path, "w") as f:
    f.write(new)

# verify
!grep -n "392\|448" /kaggle/working/GenWildSplat/src/utils/image.py

In [ ]:
import os
!/kaggle/working/miniconda/envs/genwild/bin/python -m pip install -U gdown -q

zip_path = "/kaggle/working/GenWildSplat/checkpoint_download.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

!cd /kaggle/working/GenWildSplat && /kaggle/working/miniconda/envs/genwild/bin/python -m gdown 1LNK4_NqZOjGw4OKt3cLrsRW0tH-pfnov -O checkpoint_download.zip

size_mb = os.path.getsize(zip_path) / 1e6 if os.path.exists(zip_path) else 0
print(f"\ndownloaded size: {size_mb:.1f} MB")
if size_mb < 1000:
    print("⚠️ TOO SMALL — download failed (likely Google Drive rate limit). Do NOT proceed.")
else:
    print("✅ size looks right, safe to extract")

In [ ]:
import zipfile, os, shutil

zip_path_working = "/kaggle/working/GenWildSplat/checkpoint_download.zip"
zip_path_root = "/tmp_ckpt.zip"   # on the huge 1.1TB root filesystem
ckpt_dir = "/kaggle/working/GenWildSplat/checkpoint"

# --- 1. move zip OFF the 20GB working volume onto the 1.1TB root fs ---
# (frees ~5GB in /kaggle/working so the model can be written)
if os.path.exists(zip_path_working):
    shutil.move(zip_path_working, zip_path_root)
    print("moved zip to root filesystem")

# --- 2. extract ONLY the two weight files, directly into checkpoint/ ---
with zipfile.ZipFile(zip_path_root, 'r') as z:
    names = z.namelist()
    for target in ["model.safetensors", "yolov8x-seg.pt"]:
        match = [n for n in names if n.endswith(target) and not n.startswith("__MACOSX")]
        if match:
            with z.open(match[0]) as src, open(os.path.join(ckpt_dir, target), "wb") as dst:
                shutil.copyfileobj(src, dst)
            print(f"extracted {target}")

# --- 3. delete the zip from root fs ---
os.remove(zip_path_root)
print("removed zip")

# --- 4. verify checkpoint has all 3 files ---
print("\n--- checkpoint/ ---")
for f in os.listdir(ckpt_dir):
    print(" ", f, f"{os.path.getsize(os.path.join(ckpt_dir, f))/1e6:.1f} MB")

# --- 5. RE-APPLY the 392 memory fix (wiped on re-clone) ---
img_py = "/kaggle/working/GenWildSplat/src/utils/image.py"
with open(img_py) as f:
    code = f.read()
if "448" in code:
    with open(img_py, "w") as f:
        f.write(code.replace("448", "392"))
    print("\n✅ patched 448 -> 392 (memory fix)")
else:
    print("\nℹ️ 392 already applied")

!echo "--- disk after ---" && df -h /kaggle/working

## 5. Reproduction sanity check: Oidor Chapel

This section runs the released evaluation script on the public Oidor Chapel example. The purpose is to verify that the checkpoint, dependencies, camera/geometry path, and rendering path are functioning before running additional diagnostic scenes.


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

!cd /kaggle/working/GenWildSplat && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True /kaggle/working/miniconda/envs/genwild/bin/python src/eval_nvs_tgt.py --data_dir examples/Oidor_Chapel --output_path eval_outputs/Oidor_Chapel --ckpt_path checkpoint/model.safetensors --no_refine

In [ ]:
!find /kaggle/working/GenWildSplat/eval_outputs/Oidor_Chapel -type f | head -20

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

base = "/kaggle/working/GenWildSplat/eval_outputs/Oidor_Chapel/nvs"
gt_dir = os.path.join(base, "gt")
pred_dir = os.path.join(base, "pred")

# sort so gt and pred line up by identical filename
gt_files = sorted(os.listdir(gt_dir))
pred_files = sorted(os.listdir(pred_dir))

n = len(gt_files)
fig, axes = plt.subplots(n, 2, figsize=(10, 5 * n))
if n == 1:
    axes = axes.reshape(1, 2)

for i, fname in enumerate(gt_files):
    gt_img = Image.open(os.path.join(gt_dir, fname))
    pred_img = Image.open(os.path.join(pred_dir, fname))  # same filename in both
    axes[i, 0].imshow(gt_img); axes[i, 0].set_title("Ground Truth"); axes[i, 0].axis("off")
    axes[i, 1].imshow(pred_img); axes[i, 1].set_title("Predicted"); axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/comparison.png", dpi=80, bbox_inches="tight")
plt.show()
print("saved to /kaggle/working/comparison.png")

## 6. Dataset inspection and candidate scene review

The diagnostic scenes are drawn from the Mip-NeRF 360 dataset available in the Kaggle input directory. These cells inspect the folder structure, list available images, and review the mask-generation script used later in the occlusion probe.


In [ ]:
import os
base = "/kaggle/input/datasets/thnhdg/testing"
# what scenes exist, and what's the folder structure of garden?
for root, dirs, files in os.walk(base):
    depth = root.replace(base, "").count(os.sep)
    if depth <= 2:
        print(root, "->", dirs[:8] if dirs else f"({len(files)} files)")

In [ ]:
import os
garden_imgs = "/kaggle/input/datasets/thnhdg/testing/360_v2/garden/images_8"
files = sorted(os.listdir(garden_imgs))
print(f"garden: {len(files)} images")
print("first 5:", files[:5])
print("last 5:", files[-5:])

In [ ]:
!cat /kaggle/working/GenWildSplat/create_masks.py

## 7. Diagnostic scene selection

This section selects representative frames for the nine diagnostic scenes: garden, stump, flowers, treehill, bicycle, kitchen, counter, bonsai, and room. Some cells use geometric heuristics from camera poses, while others manually inspect or override frame choices to obtain overlapping views suitable for novel-view synthesis.


In [ ]:
for scene in ["360_v2/garden", "360_v2/kitchen","360_v2/stump","360_v2/room","360_v2/bicycle" ]:
    d = f"/kaggle/input/datasets/thnhdg/testing/{scene}/images_8"
    files = sorted(os.listdir(d))
    print(f"\n{scene}: {len(files)} images")
    sel = files[:8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    for ax, fname in zip(axes.flat, sel):
        ax.imshow(Image.open(os.path.join(d, fname)))
        ax.set_title(fname, fontsize=8); ax.axis("off")
    plt.suptitle(scene); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
p = np.load("/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle/poses_bounds.npy")
print("shape:", p.shape)   # expect (N, 17)
print("first row:", p[0])

In [ ]:
import numpy as np

def get_camera_centers(poses_bounds_path):
    """Extract 3D camera positions from an LLFF poses_bounds.npy."""
    pb = np.load(poses_bounds_path)               # (N, 17)
    poses = pb[:, :15].reshape(-1, 3, 5)          # (N, 3, 5)
    centers = poses[:, :, 3]                       # (N, 3) translation = camera position
    return centers

def select_overlapping_frames(poses_bounds_path, image_files, n_select=8,
                              window_frac=0.15, verbose=True):
    """
    window_frac: fraction of the ordered trajectory to sample from.
    Smaller = tighter overlap, all frames see the same region.
    """
    centers = get_camera_centers(poses_bounds_path)
    N = len(centers)
    assert N == len(image_files)

    # greedy nearest-neighbor ordering (trajectory order)
    visited = [0]; remaining = set(range(1, N))
    while remaining:
        last = centers[visited[-1]]
        nxt = min(remaining, key=lambda i: np.linalg.norm(centers[i] - last))
        visited.append(nxt); remaining.discard(nxt)

    # take n_select frames from a TIGHT window at the start of the ordered path
    window_len = max(n_select, int(N * window_frac))
    window = visited[:window_len]
    step = max(1, window_len // n_select)
    picked_idx = window[0:step*n_select:step][:n_select]

    picked_files = [image_files[i] for i in picked_idx]
    if verbose:
        dists = [np.linalg.norm(centers[picked_idx[i+1]] - centers[picked_idx[i]])
                 for i in range(len(picked_idx)-1)]
        print(f"  {len(picked_files)} frames; baselines: {[f'{d:.2f}' for d in dists]}")
    return picked_files

In [ ]:
import os, matplotlib.pyplot as plt
from PIL import Image

scene = "360_v2/bicycle"
img_dir = f"/kaggle/input/datasets/thnhdg/testing/{scene}/images_8"
pb_path = f"/kaggle/input/datasets/thnhdg/testing/{scene}/poses_bounds.npy"

files = sorted(os.listdir(img_dir))
picked = select_overlapping_frames(pb_path, files, n_select=8)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, fname in zip(axes.flat, picked):
    ax.imshow(Image.open(os.path.join(img_dir, fname)))
    ax.set_title(fname, fontsize=8); ax.axis("off")
plt.suptitle(f"{scene} — geometry-selected frames"); plt.tight_layout(); plt.show()

In [ ]:
manual_frames = {
    "bicycle": [
        "_DSC8679.JPG",  # #0
        "_DSC8782.JPG",  # #22  ← swapped in for #3
        "_DSC8725.JPG",  # #4
        "_DSC8778.JPG",  # #5
        "_DSC8816.JPG",  # #7
        "_DSC8858.JPG",  # #9
        "_DSC8818.JPG",  # #10
        "_DSC8780.JPG",  # #12
    ],
}
import os, matplotlib.pyplot as plt
from PIL import Image

img_dir = "/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle/images_8"
picks = manual_frames["bicycle"]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, f in zip(axes.flat, picks):
    ax.imshow(Image.open(os.path.join(img_dir, f)))
    ax.set_title(f, fontsize=9); ax.axis("off")
plt.suptitle("bicycle — final picks (with #22 swap)"); plt.tight_layout(); plt.show()

In [ ]:
import os, matplotlib.pyplot as plt
from PIL import Image

img_dir = "/kaggle/input/datasets/thnhdg/testing/360_v2/stump/images_8"
stump_picks = ["_DSC9309.JPG","_DSC9328.JPG","_DSC9329.JPG","_DSC9311.JPG",
               "_DSC9273.JPG","_DSC9235.JPG","_DSC9217.JPG","_DSC9216.JPG"]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, f in zip(axes.flat, stump_picks):
    ax.imshow(Image.open(os.path.join(img_dir, f)))
    ax.set_title(f, fontsize=9); ax.axis("off")
plt.suptitle("stump — candidate picks"); plt.tight_layout(); plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from PIL import Image

def geometry_select(pb_path, files, n=8, window_frac=0.15):
    c = np.load(pb_path)[:, :15].reshape(-1,3,5)[:, :, 3]
    visited=[0]; rem=set(range(1,len(c)))
    while rem:
        last=c[visited[-1]]; nxt=min(rem,key=lambda i:np.linalg.norm(c[i]-last))
        visited.append(nxt); rem.discard(nxt)
    w=max(n,int(len(c)*window_frac)); win=visited[:w]; step=max(1,w//n)
    return [files[i] for i in win[0:step*n:step][:n]]

remaining = {
    "garden":  "360_v2/garden",
    "kitchen": "360_v2/kitchen",
    "room":    "360_v2/room",
    "counter": "360_v2/counter",
    "bonsai":  "360_v2/bonsai",
}

auto_picks = {}
for name, rel in remaining.items():
    img_dir = f"/kaggle/input/datasets/thnhdg/testing/{rel}/images_8"
    pb_path = f"/kaggle/input/datasets/thnhdg/testing/{rel}/poses_bounds.npy"
    files = sorted(os.listdir(img_dir))
    picks = geometry_select(pb_path, files)
    auto_picks[name] = picks

    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    for ax, f in zip(axes.flat, picks):
        ax.imshow(Image.open(os.path.join(img_dir, f))); ax.set_title(f, fontsize=8); ax.axis("off")
    plt.suptitle(f"{name} — auto-picked");

In [ ]:
import os, matplotlib.pyplot as plt
from PIL import Image

DATASET = "/kaggle/input/datasets/thnhdg/testing"

# scene -> folder path (to locate the images)
scene_paths = {
    "garden":   "360_v2/garden",
    "stump":    "360_v2/stump",
    "flowers":  "360_extra_scenes/flowers",
    "treehill": "360_extra_scenes/treehill",
    "bicycle":  "360_v2/bicycle",
    "kitchen":  "360_v2/kitchen",
    "counter":  "360_v2/counter",
    "bonsai":   "360_v2/bonsai",
    "room":     "360_v2/room",
}

manual_frames = {
    "garden":  ["DSC07956.JPG","DSC08054.JPG","DSC08113.JPG","DSC08026.JPG",
                "DSC07958.JPG","DSC08057.JPG","DSC08116.JPG","DSC08028.JPG"],
    "stump":   ["_DSC9309.JPG","_DSC9328.JPG","_DSC9329.JPG","_DSC9311.JPG",
                "_DSC9273.JPG","_DSC9235.JPG","_DSC9217.JPG","_DSC9216.JPG"],
    "flowers": ["_DSC9040.JPG","_DSC9078.JPG","_DSC9120.JPG","_DSC9162.JPG",
                "_DSC9160.JPG","_DSC9076.JPG","_DSC9118.JPG","_DSC9159.JPG"],
    "treehill":["_DSC8874.JPG","_DSC8902.JPG","_DSC8929.JPG","_DSC8928.JPG",
                "_DSC8980.JPG","_DSC9004.JPG","_DSC8930.JPG","_DSC8876.JPG"],
    "bicycle": ["_DSC8679.JPG","_DSC8782.JPG","_DSC8725.JPG","_DSC8778.JPG",
                "_DSC8816.JPG","_DSC8858.JPG","_DSC8818.JPG","_DSC8780.JPG"],
    "kitchen": ["DSCF0656.JPG","DSCF0661.JPG","DSCF0666.JPG","DSCF0671.JPG",
                "DSCF0676.JPG","DSCF0681.JPG","DSCF0686.JPG","DSCF0691.JPG"],
    "counter": ["DSCF5857.JPG","DSCF5861.JPG","DSCF5865.JPG","DSCF5869.JPG",
                "DSCF5873.JPG","DSCF5877.JPG","DSCF5881.JPG","DSCF5885.JPG"],
    "bonsai":  ["DSCF5565.JPG","DSCF5570.JPG","DSCF5575.JPG","DSCF5580.JPG",
                "DSCF5585.JPG","DSCF5590.JPG","DSCF5595.JPG","DSCF5600.JPG"],
    "room":    ["DSCF4667.JPG","DSCF4669.JPG","DSCF4671.JPG","DSCF4673.JPG",
                "DSCF4674.JPG","DSCF4676.JPG","DSCF4807.JPG","DSCF4817.JPG"],
}

missing = []
for name, picks in manual_frames.items():
    img_dir = f"{DATASET}/{scene_paths[name]}/images_8"
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    for ax, f in zip(axes.flat, picks):
        path = os.path.join(img_dir, f)
        if os.path.exists(path):
            ax.imshow(Image.open(path))
        else:
            ax.text(0.5, 0.5, f"MISSING\n{f}", ha="center", va="center", color="red")
            missing.append((name, f))
        ax.set_title(f, fontsize=9); ax.axis("off")
    plt.suptitle(f"{name}  ({scene_paths[name]})", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

print("\n" + "="*50)
if missing:
    print("⚠️ MISSING FILES (fix these before running):")
    for n, f in missing: print(f"  {n}: {f}")
else:
    print("✅ All 72 files (9 scenes × 8) exist and displayed.")

## 8. Probe 1: diagnostic generalization behavior

The first empirical probe evaluates GenWildSplat on the nine selected scenes using blank masks. This measures how reconstruction quality varies across scene types under a consistent inference setup.


In [ ]:
import os, shutil, subprocess, re
import numpy as np
from PIL import Image

DATASET = "/kaggle/input/datasets/thnhdg/testing"
WORK = "/kaggle/working/probe3"
GWS = "/kaggle/working/GenWildSplat"
PYBIN = "/kaggle/working/miniconda/envs/genwild/bin/python"

scene_paths = {
    "garden":"360_v2/garden","stump":"360_v2/stump",
    "flowers":"360_extra_scenes/flowers","treehill":"360_extra_scenes/treehill",
    "bicycle":"360_v2/bicycle","kitchen":"360_v2/kitchen",
    "counter":"360_v2/counter","bonsai":"360_v2/bonsai","room":"360_v2/room",
}
indoor = {"kitchen","room","counter","bonsai"}

# (manual_frames dict already defined in your notebook from the verification cell)

def make_blank_masks(img_dir, mask_dir):
    """All-zero masks (no transients in clean Mip-NeRF scenes)."""
    os.makedirs(mask_dir, exist_ok=True)
    for f in os.listdir(img_dir):
        im = Image.open(os.path.join(img_dir, f))
        Image.new("L", im.size, 0).save(os.path.join(mask_dir, f))

results = {}
for name, picks in manual_frames.items():
    print(f"\n{'='*45}\n{name}\n{'='*45}")
    src = f"{DATASET}/{scene_paths[name]}/images_8"
    dst = f"{WORK}/{name}"; img_dir=f"{dst}/images"; mask_dir=f"{dst}/masks"

    # fresh prep
    if os.path.exists(dst): shutil.rmtree(dst)
    os.makedirs(img_dir)
    for f in picks:
        shutil.copy(os.path.join(src, f), os.path.join(img_dir, f))
    make_blank_masks(img_dir, mask_dir)

    out = subprocess.run(
        [PYBIN, "src/eval_nvs_tgt.py", "--data_dir", dst,
         "--output_path", f"eval_outputs/probe3_{name}",
         "--ckpt_path", "checkpoint/model.safetensors", "--no_refine"],
        cwd=GWS, capture_output=True, text=True,
        env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})

    m = re.search(r"PSNR:\s*([\d.]+),\s*SSIM:\s*([\d.]+),\s*LPIPS:\s*([\d.]+)", out.stdout)
    if m:
        results[name] = tuple(float(x) for x in m.groups())
        print(f"✅ PSNR={m.group(1)}  SSIM={m.group(2)}  LPIPS={m.group(3)}")
    else:
        results[name] = None
        print("❌ FAILED — stderr tail:")
        print(out.stderr[-1000:])

# ---- summary table ----
print("\n\n" + "="*58)
print(f"{'Scene':<12}{'Type':<10}{'PSNR↑':>8}{'SSIM↑':>8}{'LPIPS↓':>9}")
print("="*58)
out_vals, in_vals = [], []
for name, r in results.items():
    t = "indoor" if name in indoor else "outdoor"
    if r:
        print(f"{name:<12}{t:<10}{r[0]:>8.2f}{r[1]:>8.3f}{r[2]:>9.3f}")
        (in_vals if name in indoor else out_vals).append(r)
    else:
        print(f"{name:<12}{t:<10}{'FAILED':>8}")

# group averages — the key finding
def avg(vals, i): return sum(v[i] for v in vals)/len(vals) if vals else float('nan')
print("-"*58)
if out_vals:
    print(f"{'OUTDOOR avg':<22}{avg(out_vals,0):>8.2f}{avg(out_vals,1):>8.3f}{avg(out_vals,2):>9.3f}")
if in_vals:
    print(f"{'INDOOR avg':<22}{avg(in_vals,0):>8.2f}{avg(in_vals,1):>8.3f}{avg(in_vals,2):>9.3f}")

### Visualizing diagnostic reconstructions

The next cell visualizes ground-truth target views and corresponding predictions for each diagnostic scene. These outputs support the qualitative comparison used in the report.


In [ ]:
import os, matplotlib.pyplot as plt
from PIL import Image

GWS = "/kaggle/working/GenWildSplat"
scenes = ["garden","stump","flowers","treehill","bicycle","kitchen","counter","bonsai","room"]

for name in scenes:
    base = f"{GWS}/eval_outputs/probe3_{name}/nvs"
    gt_dir, pred_dir = f"{base}/gt", f"{base}/pred"
    if not os.path.exists(gt_dir):
        print(f"⚠️ no output for {name} (did it fail?)"); continue
    files = sorted(os.listdir(gt_dir))
    n = len(files)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    if n == 1: axes = axes.reshape(2,1)
    for i, f in enumerate(files):
        axes[0,i].imshow(Image.open(os.path.join(gt_dir,f)))
        axes[0,i].set_title("GT", fontsize=8); axes[0,i].axis("off")
        axes[1,i].imshow(Image.open(os.path.join(pred_dir,f)))
        axes[1,i].set_title("Pred", fontsize=8); axes[1,i].axis("off")
    plt.suptitle(f"{name} — reconstructions (GT top, predicted bottom)", fontweight="bold")
    plt.tight_layout(); plt.show()

## 9. Probe 2: occlusion-mask sensitivity

The second empirical probe compares blank masks against YOLOv8-generated masks. This evaluates whether the semantic occlusion prior helps or hurts in relatively clean diagnostic scenes, where aggressive masking may remove static content rather than only transient objects.


In [ ]:
!/kaggle/working/miniconda/envs/genwild/bin/python -m pip install ultralytics -q

In [ ]:
import os, subprocess
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

GWS = "/kaggle/working/GenWildSplat"
PYBIN = "/kaggle/working/miniconda/envs/genwild/bin/python"
WORK = "/kaggle/working/probe3"
YOLO_CKPT = f"{GWS}/checkpoint/yolov8x-seg.pt"

# small script to run YOLO in the CONDA ENV (kernel-YOLO is broken)
mask_script = '''
import os, sys, cv2, numpy as np
from ultralytics import YOLO
img_dir, mask_dir, ckpt = sys.argv[1], sys.argv[2], sys.argv[3]
os.makedirs(mask_dir, exist_ok=True)
model = YOLO(ckpt)
tc = {0,1,2,3,5,6,7,8,14,15,16,17,18,19,20,21,22,23,24,25,26,28,56,67,73}
for f in os.listdir(img_dir):
    if not f.lower().endswith((".jpg",".jpeg",".png")): continue
    res = model(os.path.join(img_dir,f), conf=0.01, verbose=False)[0]
    h,w = res.orig_img.shape[:2]; m = np.zeros((h,w),np.uint8)
    if res.masks is not None:
        for poly,cls in zip(res.masks.xy,res.boxes.cls):
            if int(cls) in tc: cv2.fillPoly(m,[poly.reshape(-1,1,2).astype(int)],255)
    cv2.imwrite(os.path.join(mask_dir,f), m)
    print(f"{f}: {100*m.mean()/255:.2f}% masked")
'''
with open("/kaggle/working/gen_yolo_masks.py","w") as fp: fp.write(mask_script)

# run on a few scenes to check (add more if you like)
check = ["garden","bicycle","kitchen","room"]
for name in check:
    img_dir = f"{WORK}/{name}/images"
    ym_dir  = f"{WORK}/{name}/yolo_masks"
    print(f"\n=== {name}: YOLOv8-seg ===")
    r = subprocess.run([PYBIN,"/kaggle/working/gen_yolo_masks.py",img_dir,ym_dir,YOLO_CKPT],
                       capture_output=True, text=True)
    print(r.stdout[-500:])
    if r.returncode != 0: print("ERR:", r.stderr[-500:])

# display input vs YOLO mask
for name in check:
    img_dir = f"{WORK}/{name}/images"; ym_dir = f"{WORK}/{name}/yolo_masks"
    files = sorted(os.listdir(img_dir))[:4]
    fig, axes = plt.subplots(2,4,figsize=(20,8))
    for i,f in enumerate(files):
        axes[0,i].imshow(Image.open(os.path.join(img_dir,f))); axes[0,i].axis("off")
        axes[0,i].set_title(f, fontsize=7)
        mp = os.path.join(ym_dir,f)
        if os.path.exists(mp):
            m = np.array(Image.open(mp))
            axes[1,i].imshow(m, cmap="gray", vmin=0, vmax=255)
            axes[1,i].set_title(f"{100*m.mean()/255:.1f}% masked", fontsize=7)
        axes[1,i].axis("off")
    plt.suptitle(f"{name}: input (top) vs YOLOv8 mask (bottom)", fontweight="bold")
    plt.tight_layout(); plt.show()

In [ ]:
import os, shutil, subprocess, re
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

DATASET = "/kaggle/input/datasets/thnhdg/testing"
WORK = "/kaggle/working/probe3"
GWS = "/kaggle/working/GenWildSplat"
PYBIN = "/kaggle/working/miniconda/envs/genwild/bin/python"
YOLO_CKPT = f"{GWS}/checkpoint/yolov8x-seg.pt"

scene_paths = {
    "garden":"360_v2/garden","stump":"360_v2/stump",
    "flowers":"360_extra_scenes/flowers","treehill":"360_extra_scenes/treehill",
    "bicycle":"360_v2/bicycle","kitchen":"360_v2/kitchen",
    "counter":"360_v2/counter","bonsai":"360_v2/bonsai","room":"360_v2/room",
}
indoor = {"kitchen","room","counter","bonsai"}
all_scenes = list(scene_paths.keys())

# blank-mask (Probe 3) results for comparison
blank_results = {
    "garden":(16.38,0.254,0.460), "stump":(15.98,0.171,0.601),
    "flowers":(14.12,0.108,0.567), "treehill":(13.33,0.252,0.610),
    "bicycle":(14.57,0.193,0.513), "kitchen":(14.98,0.318,0.424),
    "counter":(10.09,0.319,0.607), "bonsai":(14.09,0.305,0.592),
    "room":(11.56,0.428,0.538),
}

# ---- 1. generate YOLO masks for any scene missing them (via conda env) ----
mask_script = '''
import os, sys, cv2, numpy as np
from ultralytics import YOLO
img_dir, mask_dir, ckpt = sys.argv[1], sys.argv[2], sys.argv[3]
os.makedirs(mask_dir, exist_ok=True)
model = YOLO(ckpt)
tc = {0,1,2,3,5,6,7,8,14,15,16,17,18,19,20,21,22,23,24,25,26,28,56,67,73}
for f in os.listdir(img_dir):
    if not f.lower().endswith((".jpg",".jpeg",".png")): continue
    res = model(os.path.join(img_dir,f), conf=0.01, verbose=False)[0]
    h,w = res.orig_img.shape[:2]; m = np.zeros((h,w),np.uint8)
    if res.masks is not None:
        for poly,cls in zip(res.masks.xy,res.boxes.cls):
            if int(cls) in tc: cv2.fillPoly(m,[poly.reshape(-1,1,2).astype(int)],255)
    cv2.imwrite(os.path.join(mask_dir,f), m)
    print(f"{f}: {100*m.mean()/255:.2f}% masked")
'''
with open("/kaggle/working/gen_yolo_masks.py","w") as fp: fp.write(mask_script)

for name in all_scenes:
    img_dir = f"{WORK}/{name}/images"
    ym_dir  = f"{WORK}/{name}/yolo_masks"
    # skip if masks already exist for all 8 images
    if os.path.exists(ym_dir) and len(os.listdir(ym_dir)) == len(os.listdir(img_dir)):
        print(f"✓ {name}: yolo masks already exist")
        continue
    print(f"→ generating YOLO masks for {name}...")
    r = subprocess.run([PYBIN,"/kaggle/working/gen_yolo_masks.py",img_dir,ym_dir,YOLO_CKPT],
                       capture_output=True, text=True)
    if r.returncode != 0: print(f"  ❌ {name}:", r.stderr[-400:])

# ---- 2. run eval with YOLO masks on all 9 ----
def run_with_yolo_masks(name):
    src_img = f"{WORK}/{name}/images"
    ym_dir  = f"{WORK}/{name}/yolo_masks"
    dst = f"{WORK}/{name}_yolo"
    img_dir, mask_dir = f"{dst}/images", f"{dst}/masks"
    if os.path.exists(dst): shutil.rmtree(dst)
    os.makedirs(img_dir); os.makedirs(mask_dir)
    for f in os.listdir(src_img):
        shutil.copy(f"{src_img}/{f}", f"{img_dir}/{f}")
        ym = os.path.join(ym_dir, f)
        if os.path.exists(ym):
            shutil.copy(ym, f"{mask_dir}/{f}")
        else:  # fallback blank if a mask is somehow missing
            Image.new("L", Image.open(f"{img_dir}/{f}").size, 0).save(f"{mask_dir}/{f}")
    out = subprocess.run(
        [PYBIN,"src/eval_nvs_tgt.py","--data_dir",dst,
         "--output_path",f"eval_outputs/probe2_{name}",
         "--ckpt_path","checkpoint/model.safetensors","--no_refine"],
        cwd=GWS, capture_output=True, text=True,
        env={**os.environ,"PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})
    m = re.search(r"PSNR:\s*([\d.]+),\s*SSIM:\s*([\d.]+),\s*LPIPS:\s*([\d.]+)", out.stdout)
    return (tuple(float(x) for x in m.groups()) if m else None), out

yolo_results = {}
for name in all_scenes:
    print(f"\n=== {name}: eval WITH YOLO masks ===")
    r, out = run_with_yolo_masks(name)
    yolo_results[name] = r
    if r: print(f"✅ PSNR={r[0]:.2f} SSIM={r[1]:.3f} LPIPS={r[2]:.3f}")
    else: print("❌ FAILED:", out.stderr[-400:])

# ---- 3. comparison table: blank vs YOLO ----
print("\n\n" + "="*78)
print(f"{'Scene':<10}{'Type':<9}{'blank PSNR':>11}{'YOLO PSNR':>11}{'ΔPSNR':>9}{'blank LPIPS':>13}{'YOLO LPIPS':>12}")
print("="*78)
for name in all_scenes:
    b = blank_results[name]; y = yolo_results.get(name)
    t = "indoor" if name in indoor else "outdoor"
    if y:
        print(f"{name:<10}{t:<9}{b[0]:>11.2f}{y[0]:>11.2f}{y[0]-b[0]:>+9.2f}{b[2]:>13.3f}{y[2]:>12.3f}")
    else:
        print(f"{name:<10}{t:<9}{b[0]:>11.2f}{'FAIL':>11}")

# ---- 4. visualize: GT / blank-pred / YOLO-pred (top 2 target views) ----
for name in all_scenes:
    bb = f"{GWS}/eval_outputs/probe3_{name}/nvs"
    yb = f"{GWS}/eval_outputs/probe2_{name}/nvs"
    if not (os.path.exists(f"{bb}/pred") and os.path.exists(f"{yb}/pred")): continue
    files = sorted(os.listdir(f"{bb}/gt"))[:2]
    fig, axes = plt.subplots(3, len(files), figsize=(5*len(files), 11))
    if len(files)==1: axes = axes.reshape(3,1)
    for i,f in enumerate(files):
        axes[0,i].imshow(Image.open(f"{bb}/gt/{f}")); axes[0,i].set_title("GT",fontsize=9); axes[0,i].axis("off")
        axes[1,i].imshow(Image.open(f"{bb}/pred/{f}")); axes[1,i].set_title("Pred (blank mask)",fontsize=9); axes[1,i].axis("off")
        axes[2,i].imshow(Image.open(f"{yb}/pred/{f}")); axes[2,i].set_title("Pred (YOLO mask)",fontsize=9); axes[2,i].axis("off")
    plt.suptitle(f"{name}: blank-mask vs YOLO-mask reconstruction", fontweight="bold")
    plt.tight_layout(); plt.show()

## 10. Attempted probe: light-code pathway tracing

This section traces how target lighting information flows through the released implementation. The goal is to identify what would be required for a clean light-code disentanglement experiment, such as swapping or interpolating light codes while holding geometry fixed. The cells inspect relevant files and symbols rather than claiming a completed intervention.


In [ ]:
!ls /kaggle/working/GenWildSplat/src/model/encoder/
!echo "=== looking for light encoder / appearance adapter ==="
!grep -rn "class.*Light\|def.*light\|appearance\|Flight\|ELight\|light_code\|lgt" /kaggle/working/GenWildSplat/src/model/encoder/*.py | head -40

In [ ]:
# how does the eval script build target_lgt_info?
!grep -n "target_lgt_info\|tgt_images\|lgt" /kaggle/working/GenWildSplat/src/eval_nvs_tgt.py

# and how the aggregator uses it
!grep -n "target_lgt_info\|lgt_embed\|light" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py | head -30

In [ ]:
# 1. In the aggregator: where does global_lighting come from and get used?
print("="*30, "AGGREGATOR", "="*30)
!grep -n "global_lighting\|lgt_embed\|light_embed\|lighting" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py

In [ ]:
# 2. Find the module/function that ENCODES images -> 16-dim light code
print("="*30, "LIGHT ENCODER SEARCH", "="*30)
!grep -rn "16\|light_embed_dim\|lighting_dim\|LightEncoder\|encode.*lgt\|lgt.*encode\|nn.Linear.*16\|Conv.*light" /kaggle/working/GenWildSplat/src/model/encoder/vggt/ | grep -i "light\|lgt\|16" | head -40

In [ ]:
# 3. Look at the aggregator's forward signature + how target_lgt_info flows to global_lighting
print("="*30, "FORWARD FLOW", "="*30)
!sed -n '430,545p' /kaggle/working/GenWildSplat/src/model/encoder/anysplat.py

In [ ]:
# Where is this FiLM module CALLED, and what's passed as global_lighting?
!grep -rn "global_lighting\|lighting_to_gamma\|LightingModulation\|FiLM\|light_modul" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py


In [ ]:
# The whole file that contains the FiLM module (lines 1-60) + find its class name
!sed -n '1,60p' /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py


In [ ]:
# Trace: where target_lgt_info (images) becomes the 16-dim code
!grep -rn "global_lighting\s*=\|lgt.*=.*encod\|light.*=.*(\|=.*lighting_encoder\|context_gt_lgt_embeds" /kaggle/working/GenWildSplat/src/model/encoder/anysplat.py

In [ ]:
# find where GlobalLightingInjection is created and used
!grep -n "GlobalLightingInjection\|lighting_injection\|self.lighting\|light_inject" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py

# see how it's called in the aggregator forward (what's passed as global_lighting)
!grep -n "global_lighting\|lighting_injection\|\.lighting" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py

In [ ]:
# 1. Is there ANOTHER lighting module instance, or where does the 16-dim LIGHT code actually go?
!grep -n "lighting_injection\|light_inject\|GlobalLighting\|lgt_embed\|lighting_fuser\|self.light\|light_net\|context_gt_lgt" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py

# 2. Trace target_lgt_info THROUGH the aggregator — where does it get turned into a code and injected?
!grep -n "target_lgt_info\|lgt\|lighting\|global_light" /kaggle/working/GenWildSplat/src/model/encoder/vggt/models/aggregator.py

## 11. Summary

This notebook provides the computational record behind the accompanying report. The main completed components are the local reproduction sanity check, the nine-scene diagnostic evaluation, and the blank-mask versus YOLO-mask sensitivity analysis. The light-code cells document implementation tracing for future work, including possible light-code swaps and interpolation experiments.
